In [0]:
-- ============================================================
-- NOTEBOOK: 01_bronze_ctas_ingestion.sql
-- PURPOSE : Read CSV files from Unity Catalog Volume and land
--           them as-is into Bronze Delta tables using CTAS.
-- CATALOG STRUCTURE:
--   dev.bronze.raw_*   |  uat.bronze.raw_*  |  prod.bronze.raw_*
-- VOLUME  : /{catalog}/bronze/raw/
-- AUTHOR  : The Data Master | Naval Yemul
-- ============================================================

-- ─────────────────────────────────────────────────────────────
-- WIDGET PARAMETERS  (set at Lakeflow Job task level)
-- ─────────────────────────────────────────────────────────────
-- env  = dev | uat | prod
--
-- In the Python cell above this SQL cell:
--
--   env          = dbutils.widgets.get("env")          # dev | uat | prod
--   catalog      = env                                  # catalog name = env name
--   volume_path  = f"/Volumes/{catalog}/bronze/raw"
--
--   spark.conf.set("pipeline.catalog",      catalog)
--   spark.conf.set("pipeline.volume_path",  volume_path)
--
-- All SQL cells below use :catalog and :volume_path
-- which Databricks resolves from spark.conf at runtime.
-- In a Lakeflow Job SQL task use:
--   {{job.parameters.env}} directly.


-- ─────────────────────────────────────────────────────────────
-- STEP 1 : CTAS — raw_transactions
-- ─────────────────────────────────────────────────────────────

CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.bronze.raw_transactions')
  USING DELTA
  TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true',
    'pipeline.layer'                   = 'bronze',
    'pipeline.source'                  = 'volume_csv',
    'pipeline.load_type'               = 'full',
    'pipeline.loaded_by'               = 'naval.yemul'
  )
  COMMENT 'Bronze: raw financial transactions — no transformations applied.'
AS
SELECT
  transaction_id,
  account_id,
  customer_id,
  transaction_date,
  transaction_time,
  amount,
  currency,
  transaction_type,
  merchant_category,
  merchant_name,
  channel,
  status,
  reference_number,
  description,
  is_flagged,
  input_file_name()   AS _source_file,
  current_timestamp() AS _ingested_at,
  'full'              AS _load_type,
  :catalog            AS _env
FROM read_files(
  :volume_path || '/raw_transactions.csv',
  format      => 'csv',
  header      => true,
  inferSchema => true,
  nullValue   => '',
  mode        => 'PERMISSIVE'
);


-- ─────────────────────────────────────────────────────────────
-- STEP 2 : CTAS — raw_customers
-- ─────────────────────────────────────────────────────────────

CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.bronze.raw_customers')
  USING DELTA
  TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true',
    'pipeline.layer'                   = 'bronze',
    'pipeline.source'                  = 'volume_csv',
    'pipeline.load_type'               = 'full',
    'pipeline.loaded_by'               = 'naval.yemul'
  )
  COMMENT 'Bronze: raw customer master data — no transformations applied.'
AS
SELECT
  customer_id,
  full_name,
  email,
  phone,
  country,
  account_type,
  credit_score,
  annual_income_usd,
  created_date,
  is_active,
  input_file_name()   AS _source_file,
  current_timestamp() AS _ingested_at,
  'full'              AS _load_type,
  :catalog            AS _env
FROM read_files(
  :volume_path || '/raw_customers.csv',
  format      => 'csv',
  header      => true,
  inferSchema => true,
  nullValue   => '',
  mode        => 'PERMISSIVE'
);


-- ─────────────────────────────────────────────────────────────
-- STEP 3 : CTAS — raw_accounts
-- ─────────────────────────────────────────────────────────────

CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.bronze.raw_accounts')
  USING DELTA
  TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true',
    'pipeline.layer'                   = 'bronze',
    'pipeline.source'                  = 'volume_csv',
    'pipeline.load_type'               = 'full',
    'pipeline.loaded_by'               = 'naval.yemul'
  )
  COMMENT 'Bronze: raw account details — no transformations applied.'
AS
SELECT
  account_id,
  customer_id,
  account_type,
  currency,
  balance,
  credit_limit,
  branch_code,
  open_date,
  last_transaction_date,
  status,
  input_file_name()   AS _source_file,
  current_timestamp() AS _ingested_at,
  'full'              AS _load_type,
  :catalog            AS _env
FROM read_files(
  :volume_path || '/raw_accounts.csv',
  format      => 'csv',
  header      => true,
  inferSchema => true,
  nullValue   => '',
  mode        => 'PERMISSIVE'
);


-- ─────────────────────────────────────────────────────────────
-- STEP 4 : CTAS — raw_loans
-- ─────────────────────────────────────────────────────────────

CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.bronze.raw_loans')
  USING DELTA
  TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true',
    'pipeline.layer'                   = 'bronze',
    'pipeline.source'                  = 'volume_csv',
    'pipeline.load_type'               = 'full',
    'pipeline.loaded_by'               = 'naval.yemul'
  )
  COMMENT 'Bronze: raw loan records — no transformations applied.'
AS
SELECT
  loan_id,
  customer_id,
  account_id,
  loan_type,
  principal_amount,
  outstanding_balance,
  interest_rate_pct,
  tenure_months,
  emi_amount,
  disbursement_date,
  next_due_date,
  overdue_days,
  status,
  collateral_type,
  input_file_name()   AS _source_file,
  current_timestamp() AS _ingested_at,
  'full'              AS _load_type,
  :catalog            AS _env
FROM read_files(
  :volume_path || '/raw_loans.csv',
  format      => 'csv',
  header      => true,
  inferSchema => true,
  nullValue   => '',
  mode        => 'PERMISSIVE'
);


-- ─────────────────────────────────────────────────────────────
-- STEP 5 : VALIDATION — row counts + null spot-check
-- ─────────────────────────────────────────────────────────────

SELECT 'raw_transactions' AS table_name,
       COUNT(*)            AS total_rows,
       SUM(CASE WHEN transaction_id    IS NULL THEN 1 ELSE 0 END) AS null_pk,
       SUM(CASE WHEN amount            IS NULL THEN 1 ELSE 0 END) AS null_amount,
       SUM(CASE WHEN merchant_category IS NULL THEN 1 ELSE 0 END) AS null_merchant_cat
FROM IDENTIFIER(:catalog || '.bronze.raw_transactions')

UNION ALL
SELECT 'raw_customers', COUNT(*),
       SUM(CASE WHEN customer_id       IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN email             IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN phone             IS NULL THEN 1 ELSE 0 END)
FROM IDENTIFIER(:catalog || '.bronze.raw_customers')

UNION ALL
SELECT 'raw_accounts', COUNT(*),
       SUM(CASE WHEN account_id  IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN balance     IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN branch_code IS NULL THEN 1 ELSE 0 END)
FROM IDENTIFIER(:catalog || '.bronze.raw_accounts')

UNION ALL
SELECT 'raw_loans', COUNT(*),
       SUM(CASE WHEN loan_id             IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN outstanding_balance IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN collateral_type     IS NULL THEN 1 ELSE 0 END)
FROM IDENTIFIER(:catalog || '.bronze.raw_loans');